# Modelo QUBO/Ising para el QAP de puertas

**Archivo:** `src/qap/model.py` (propietario: Diego)

El problema se formula como:

$$
C(x)=\sum_{i<k}\sum_{j\neq l} F_{ik}D_{jl}x_{ij}x_{kl},\quad x\in\{0,1\}^{n\times n},\;\sum_j x_{ij}=1,\;\sum_i x_{ij}=1.
$$

Variables binarias $x_{i,j}=1$ si el vuelo $i$ va a la puerta $j$. Restricciones: cada vuelo a una puerta, cada puerta a un solo vuelo (matriz de permutación).

## 1) Construir el `QuadraticProgram`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
from qap.model import build_qp, to_qubo, to_ising, decode, decode_x, x_from_bits, penalty_threshold, is_feasible, true_cost, penalized_energy
from qap.types import Instance, cost

F = np.array([[0,10,4,8],[0,0,6,0],[0,0,0,0],[0,0,0,0]])
D = np.array([[0,1,2,3],[1,0,1,2],[2,1,0,1],[3,2,1,0]])
inst3 = Instance('test3', 3, F[:3,:3], D[:3,:3])
inst4 = Instance('test4', 4, F, D)

qp = build_qp(inst4)
print(qp.prettyprint())

## 2) Pasar a QUBO con penalización

In [ ]:
lam_star = penalty_threshold(inst4)
print('lambda* (umbral exacto):', lam_star)
qubo = to_qubo(qp, lam=lam_star * 2)
print('Variables en QUBO:', qubo.get_num_vars())

## 3) Pasar a Ising (Hamiltoniano)

Con la transformación $x_i = \frac{1 - Z_i}{2}$, la energía QUBO se convierte en $\langle H\rangle + offset$.

`to_ising` devuelve `(SparsePauliOp, offset)`. Se comprueba que $\langle Z\ldots Z|H|Z\ldots Z\rangle + offset = E_{QUBO}(x)$ con la correspondencia correcta de bits.

In [ ]:
op, offset = to_ising(qubo)
print('Qubits:', op.num_qubits)
print('Términos Pauli:', len(op))
print('Offset:', offset)

In [ ]:
from qiskit.quantum_info import Statevector
import itertools

for perm in list(itertools.permutations(range(4)))[:5]:
    x = np.zeros(16)
    for i, p in enumerate(perm):
        x[i*4 + p] = 1
    sv = Statevector.from_label(''.join(str(int(b)) for b in x[::-1]))  # little-endian (qubit 0 a la derecha)
    print('perm', perm, 'cost', true_cost(perm, inst4), 'QUBO', penalized_energy(x, qubo), 'Ising', float(np.real_if_close(sv.expectation_value(op)))+offset)

## 4) Decodificación y orden de bits

**Trampa importante:** Qiskit mide y devuelve las cadenas con el bit menos significativo a la derecha (little-endian), es decir el carácter más a la derecha corresponde al qubit 0. Antes de mapear a $x_{i,j}$ hay que **invertir** la cadena.

In [ ]:
x_id = np.eye(3).ravel().astype(int)
bits = ''.join(str(int(b)) for b in x_id[::-1])  # invertir para little-endian
print('bits:', bits)
print('perm:', decode(bits, 3))
print('x_from_bits:', x_from_bits(bits))

## 5) Verificación: el mínimo factible es el óptimo con λ suficiente

In [ ]:
for inst, opt in [(inst3, 24), (inst4, 32)]:
    q = to_qubo(build_qp(inst), lam=penalty_threshold(inst)*2)
    n = inst.n
    vectors = ((np.arange(2**(n*n))[:,None]) >> np.arange(n*n)) & 1
    energies = np.array([q.objective.evaluate(v.astype(int)) for v in vectors])
    p = decode_x(vectors[int(np.argmin(energies))], n)
    print(n, 'perm:', p, 'cost:', cost(p, inst), 'optimo:', opt, 'feasible:', is_feasible(p,n))

## Funciones expuestas en `model.py`

- `build_qp(inst)` → `QuadraticProgram` con variables binarias $x_{i,j}$, objetivo cuadrático $F_{ik}D_{jl}$ para $i<k, j\neq l$ y restricciones de fila/columna.
- `to_qubo(qp, lam=None)` → `QuadraticProgram` con restricciones penalizadas (`QuadraticProgramToQubo`).
- `to_ising(qubo)` → `(SparsePauliOp, offset)`. $E_{QUBO}(x)=\langle H\rangle + offset$.
- `decode(bitstring, n)` / `decode_x(x, n)`: convierten a permutación; devuelven `None` si no es matriz de permutación.
- `x_from_bits(bitstring)`: convierte bitstring little-endian al vector $x$ en orden $i*n+j$.
- `is_feasible`, `true_cost` (usa `cost()`), `penalized_energy(x, qubo)`.
- `penalty_threshold(inst)`: calcula $λ^* = \max_{x\,infeasible}\frac{C_{opt}-C(x)}{P(x)}$ por enumeración exacta para $n\le4$.